# Sarcasm Detection â€” Fine-tune DistilBERT (Google Colab / GPU)

This notebook fine-tunes **DistilBERT** on a **combined sarcasm dataset** â€” **Sarcasm Headlines v2** (news headlines) + **iSarcasmEval** SemEval-2022 Task 6 (English tweets) + **SARC Reddit** (self-annotated Reddit comments) + **MUStARD** (TV dialogue) â€” to classify text as `sarcastic` / `not_sarcastic` using a GPU.

It is a **self-contained** version of the project's training pipeline â€” it downloads its own data, so you do **not** need to upload anything.

**Runtime setup:** `Runtime â†’ Change runtime type â†’ T4 GPU` (or better).

**Output:**
- `metrics.json` â€” accuracy / precision / recall / F1 (+ target check)
- `confusion_matrix.png` â€” heatmap on the test split
- `checkpoint/` â€” the saved fine-tuned model (zip it for use in the local app)

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn pandas evaluate

## 2. Configuration

All tunables, matching the project `config.yaml`.

In [ ]:
import os
import json
import random

import numpy as np
import torch
import pandas as pd

# ---------- Config ----------
HEADLINES_URL = "https://raw.githubusercontent.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection/master/Sarcasm_Headlines_Dataset.json"
HEADLINES_PATH = "Sarcasm_Headlines_Dataset.json"
ISARCASM_URL = "https://raw.githubusercontent.com/iabufarha/iSarcasmEval/main/train/train.En.csv"
ISARCASM_PATH = "iSarcasmEval_train.En.csv"
MUSTRAD_URL = "https://raw.githubusercontent.com/Himanshu-sudo/MUStARD-dataset/master/data/sarcasm_data.json"
MUSTRAD_PATH = "muSTARD_sarcasm_data.json"
SARC_REPO = "marcbishara/sarcasm-on-reddit"
SARC_SPLIT = "sft_train"
SARC_CAP = 30000

TEXT_COL = "text"
LABEL_COL = "label"
SOURCE_COL = "source"
CLASSES = {0: "not_sarcastic", 1: "sarcastic"}

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_LABELS = 2
CHECKPOINT_DIR = "checkpoint"

TRAIN_SUBSET = 49000
VAL_SUBSET = 6000
TEST_SUBSET = 6000
BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
TARGET_F1 = 0.85

# ---------- Reproducibility ----------
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Download & prepare the data

Combines **four** sources into one unified `text` / `label` / `source` frame:
- **Sarcasm Headlines v2** (news headlines)
- **iSarcasmEval** SemEval-2022 Task 6 (English tweets)
- **SARC Reddit** (self-annotated Reddit comments, English)
- **MUStARD** (TV-dialogue utterances)

Then cleans (dedupe) and builds a **stratified** train/val/test split preserving the sarcastic share.

In [ ]:
import urllib.request
from sklearn.model_selection import train_test_split

# ---- 1) Sarcasm Headlines v2 ----
if not os.path.exists(HEADLINES_PATH):
    print("Downloading Sarcasm Headlines v2...")
    urllib.request.urlretrieve(HEADLINES_URL, HEADLINES_PATH)

rows = []
with open(HEADLINES_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
headlines = pd.DataFrame(rows)[["headline", "is_sarcastic"]].rename(
    columns={"headline": TEXT_COL, "is_sarcastic": LABEL_COL}
)
headlines[SOURCE_COL] = "headlines_v2"
print("Headlines rows:", len(headlines))

# ---- 2) iSarcasmEval (English tweets) ----
if not os.path.exists(ISARCASM_PATH):
    print("Downloading iSarcasmEval train.En.csv...")
    urllib.request.urlretrieve(ISARCASM_URL, ISARCASM_PATH)

tweets = pd.read_csv(ISARCASM_PATH)[["tweet", "sarcastic"]].rename(
    columns={"tweet": TEXT_COL, "sarcastic": LABEL_COL}
)
tweets[SOURCE_COL] = "isarcasm"
print("iSarcasmEval rows:", len(tweets))

# ---- 3) SARC Reddit (English) ----
from datasets import load_dataset as hf_load_dataset
sarc = hf_load_dataset(SARC_REPO, split=SARC_SPLIT).to_pandas()
sarc = sarc.rename(columns={"comment": TEXT_COL})[[TEXT_COL, LABEL_COL]]
sarc[SOURCE_COL] = "sarc_reddit"
if len(sarc) > SARC_CAP:
    sarc = sarc.sample(n=SARC_CAP, random_state=SEED)
print("SARC Reddit rows:", len(sarc))

# ---- 4) MUStARD (TV dialogue) ----
if not os.path.exists(MUSTRAD_PATH):
    print("Downloading MUStARD sarcasm_data.json...")
    urllib.request.urlretrieve(MUSTRAD_URL, MUSTRAD_PATH)
with open(MUSTRAD_PATH, encoding="utf-8") as f:
    mustard = pd.DataFrame(json.load(f).values())
mustard = mustard[["utterance", "sarcasm"]].rename(
    columns={"utterance": TEXT_COL, "sarcasm": LABEL_COL}
)
mustard[LABEL_COL] = mustard[LABEL_COL].astype(str).str.lower().map(
    {"true": 1, "false": 0}
).fillna(mustard[LABEL_COL]).astype(int)
mustard[SOURCE_COL] = "mu_stard"
print("MUStARD rows:", len(mustard))

# ---- Combine & clean ----
df = pd.concat([headlines, tweets, sarc, mustard], ignore_index=True)
df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df[LABEL_COL] = df[LABEL_COL].astype(int)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].ne("")]
df = df.drop_duplicates(subset=[TEXT_COL], keep="first").reset_index(drop=True)
print("Combined rows:", len(df))
print("By source:", df[SOURCE_COL].value_counts().to_dict())
print("Global sarcastic share:", round(df[LABEL_COL].mean(), 4))

# ---- Stratified split (by label) ----
y = df[LABEL_COL]
train, rem = train_test_split(df, test_size=VAL_SUBSET + TEST_SUBSET, stratify=y, random_state=SEED)
val, test = train_test_split(rem, test_size=TEST_SUBSET, stratify=rem[LABEL_COL], random_state=SEED)
train = train.sample(n=min(TRAIN_SUBSET, len(train)), random_state=SEED)
train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

for name, part in (("train", train), ("val", val), ("test", test)):
    print(f"{name}: {len(part)} rows, sarcastic share {part[LABEL_COL].mean():.3f}")

## 4. Tokenize & build datasets

In [ ]:
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tokenizer(batch[TEXT_COL], padding="max_length", truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = batch[LABEL_COL]
    return enc

def make_dataset(pdf: pd.DataFrame):
    ds = HFDataset.from_pandas(pdf[[TEXT_COL, LABEL_COL]])
    return ds.map(tokenize, batched=True, remove_columns=[TEXT_COL, LABEL_COL])

train_ds = make_dataset(train)
val_ds = make_dataset(val)
test_ds = make_dataset(test)
print("train", len(train_ds), "val", len(val_ds), "test", len(test_ds))

## 5. Fine-tune with HuggingFace `Trainer` (GPU)

Uses mixed-precision (`fp16`) on GPU, AdamW, linear warmup + decay, and computes F1 during validation. The best checkpoint is saved automatically.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import f1_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": float(f1_score(labels, preds))}

total_steps = (len(train_ds) // BATCH_SIZE + 1) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Evaluate on the held-out test set

Computes **accuracy / precision / recall / F1** and the **confusion matrix**, then checks against the project's `TARGET_F1`.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

preds = trainer.predict(test_ds)
logits = preds.predictions
trues = preds.label_ids
y_pred = np.argmax(logits, axis=-1)

accuracy = accuracy_score(trues, y_pred)
precision = precision_score(trues, y_pred)
recall = recall_score(trues, y_pred)
f1 = f1_score(trues, y_pred)
cm = confusion_matrix(trues, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print()
print(classification_report(trues, y_pred, target_names=list(CLASSES.values()), digits=4))
print("TARGET F1:", TARGET_F1, "->", "PASS" if f1 >= TARGET_F1 else "BELOW TARGET")

metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "test_examples": int(len(trues)),
    "target_f1": TARGET_F1,
    "confusion_matrix": cm.tolist(),
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json")

## 7. Confusion matrix plot + error analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Confusion matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=list(CLASSES.values()), yticklabels=list(CLASSES.values()))
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# Error analysis: misclassified examples
test.reset_index(drop=True, inplace=True)
errors = test[trues != y_pred].copy()
errors["predicted"] = [y_pred[i] for i in errors.index]
errors["actual"] = [trues[i] for i in errors.index]
errors.to_csv("error_analysis.csv", index=False)
print(f"{len(errors)}/{len(trues)} misclassified -> error_analysis.csv")
errors.head(10)

## 8. Save the model + download

Saves tokenizer + model to `checkpoint/`, then zips it so you can download it and drop it into the **local app** at `models/checkpoint/`.

In [ ]:
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print("Saved checkpoint to", CHECKPOINT_DIR)

!rm -f checkpoint.zip
!zip -r -q checkpoint.zip checkpoint
print("Created checkpoint.zip")

from google.colab import files
files.download("checkpoint.zip")
files.download("metrics.json")